In [142]:
from google.colab import drive
import pandas as pd
import numpy as np

mpesa_transactions_anomaly = drive.mount("/content/drive/")
mpesa_transactions_anomaly = "/content/drive/MyDrive/mpesa_synthetic.csv"
try:
    mpesa_transactions_anomaly = pd.read_csv(mpesa_transactions_anomaly)
    print("Success reading mpesa_transactions_anomaly")
except Exception as e:
    print(type(e).__name__, "-", str(e))

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
Success reading mpesa_transactions_anomaly


In [143]:
print("Rows loaded:", len(mpesa_transactions_anomaly))
print(mpesa_transactions_anomaly.head())

Rows loaded: 120000
  transaction_id   amount  sender_balance_before  sender_balance_after  \
0      UA0IFD0TV   703.90               58586.32              57882.42   
1      UJAHXTHV3   254.44                8088.00               7833.56   
2      UEF8MDD4V   609.04               56675.00              56065.96   
3      UBT3W5UZB  5255.34               75090.36              69835.02   
4      UGKWNNHJ7  7282.67               24408.96              17126.29   

   receiver_balance_before  receiver_balance_after transaction_type  hour  \
0                 29932.92                30636.82             peer    14   
1                 22962.44                23216.88             peer    18   
2                  1029.22                 1638.26             till     7   
3                    38.94                 5294.28          paybill    11   
4                 26237.82                33520.49             till     0   

   month_2026 day_of_week device_type   region  is_fraud  
0           1

In [144]:
d = mpesa_transactions_anomaly.copy()
d["gap_nonzero"] = (d["sender_balance_before"] - d["amount"] - d["sender_balance_after"]).abs() > 0.01
d["overdraw"] = d["amount"] > d["sender_balance_before"]
print(pd.crosstab(d["is_fraud"], d["gap_nonzero"], margins=True))
print(pd.crosstab(d["is_fraud"], d["overdraw"], margins=True))

gap_nonzero   False  True     All
is_fraud                         
0            116490     0  116490
1              1439  2071    3510
All          117929  2071  120000
overdraw   False  True     All
is_fraud                      
0         116490     0  116490
1           1439  2071    3510
All       117929  2071  120000


In [145]:
df = mpesa_transactions_anomaly.copy()
print("Fraud rate:", round(df["is_fraud"].mean(), 4))
for c in ["transaction_type", "device_type", "region", "day_of_week"]:
    print(df.groupby(c)["is_fraud"].agg(["mean", "count"]).round(4).sort_values("mean", ascending=False), "\n")
sender_gap = df["sender_balance_before"] - df["amount"] - df["sender_balance_after"]
receiver_gap = df["receiver_balance_before"] + df["amount"] - df["receiver_balance_after"]
print("Max |gap|:", round(sender_gap.abs().max(), 2), round(receiver_gap.abs().max(), 2))
print(df.groupby("is_fraud")[["amount", "sender_balance_before", "receiver_balance_before", "hour"]].agg(["mean", "median"]).T.round(1))

Fraud rate: 0.0292
                    mean  count
transaction_type               
peer              0.0298  39933
paybill           0.0294  39884
till              0.0286  40183 

               mean  count
device_type               
smartphone   0.0298  59659
feature      0.0287  60341 

           mean  count
region                
Nairobi  0.0301  24170
Nakuru   0.0299  24334
Eldoret  0.0289  23880
Mombasa  0.0289  23743
Kisumu   0.0284  23873 

               mean  count
day_of_week               
Tue          0.0309  17110
Sun          0.0305  17190
Wed          0.0299  16989
Mon          0.0291  17408
Thu          0.0288  16822
Sat          0.0281  17142
Fri          0.0276  17339 

Max |gap|: 12218.98 0.0
is_fraud                              0        1
amount                  mean     1475.8   2535.0
                        median   1019.3   2001.0
sender_balance_before   mean    40826.0  14442.2
                        median  40975.2   2350.3
receiver_balance_before mean    

In [146]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score, average_precision_score

def add_features(d):
    d = d.copy()
    d["sender_gap"] = d["sender_balance_before"] - d["amount"] - d["sender_balance_after"]
    d["receiver_gap"] = d["receiver_balance_before"] + d["amount"] - d["receiver_balance_after"]
    d["amount_ratio"] = d["amount"] / (d["sender_balance_before"] + 1)
    d["drained"] = (d["sender_balance_after"] <= 0).astype(float)
    d["hour_sin"] = np.sin(2 * np.pi * d["hour"] / 24)
    d["hour_cos"] = np.cos(2 * np.pi * d["hour"] / 24)
    for c in ["sender_gap", "receiver_gap"]:
        d[c] = np.sign(d[c]) * np.log1p(d[c].abs())
    for c in ["amount", "sender_balance_before", "sender_balance_after",
              "receiver_balance_before", "receiver_balance_after"]:
        d[c] = np.log1p(d[c].clip(lower=0))
    return d

num_cols = ["amount", "sender_balance_before", "sender_balance_after", "receiver_balance_before",
            "receiver_balance_after", "amount_ratio", "drained", "sender_gap", "receiver_gap",
            "hour_sin", "hour_cos"]
cat_cols = ["transaction_type", "device_type", "region", "day_of_week"]

df = mpesa_transactions_anomaly.copy()
X_all = pd.get_dummies(add_features(df)[num_cols + cat_cols], columns=cat_cols, drop_first=True, dtype=float)
y_all = df["is_fraud"].astype(bool).to_numpy()

m = df["month_2026"]
tr, va, te = (m <= 8).to_numpy(), m.isin([9, 10]).to_numpy(), (m >= 11).to_numpy()
X_train, X_val, X_test = X_all[tr], X_all[va], X_all[te]
y_train, y_val, y_test = y_all[tr], y_all[va], y_all[te]

X_train_normal = X_train[~y_train]
keep = X_train_normal.columns
scaler = StandardScaler().fit(X_train_normal[keep])
Xn, Xv, Xt = (scaler.transform(a[keep]) for a in (X_train_normal, X_val, X_test))
print("Dropped constant columns:", [c for c in X_all.columns if c not in keep])
print(f"Features {len(keep)} | train normal {len(Xn)} | val {len(Xv)} (fraud {y_val.sum()}) | test {len(Xt)} (fraud {y_test.sum()})")

Dropped constant columns: []
Features 24 | train normal 77775 | val 20023 (fraud 580) | test 19847 (fraud 575)


In [147]:
from sklearn.ensemble import HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(max_iter=200, class_weight="balanced", random_state=0).fit(X_train[keep], y_train)
print("Supervised test PR-AUC:", round(average_precision_score(y_test, clf.predict_proba(X_test[keep])[:, 1]), 3),
      "| base rate:", round(y_test.mean(), 3))

Supervised test PR-AUC: 0.697 | base rate: 0.029


In [148]:
from sklearn.decomposition import PCA
def pca_scores(p, Z): return np.mean((Z - p.inverse_transform(p.transform(Z))) ** 2, axis=1)

rows = []
for var in [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    p_ = PCA(n_components=var, svd_solver="full").fit(Xn)
    rows.append({"variance": var, "components": p_.n_components_,
                 "val_pr_auc": average_precision_score(y_val, pca_scores(p_, Xv))})
sweep = pd.DataFrame(rows); print(sweep.round(3))
best_var = sweep.loc[sweep.val_pr_auc.idxmax(), "variance"]
pca = PCA(n_components=best_var, svd_solver="full").fit(Xn)
val_pca, test_pca = pca_scores(pca, Xv), pca_scores(pca, Xt)

def report(name, val_s, test_s):
    p, r, t = precision_recall_curve(y_val, val_s)
    f1 = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
    th = t[np.argmax(f1)]; pred = test_s >= th
    print(f"{name}: P {precision_score(y_test, pred, zero_division=0):.4f} R {recall_score(y_test, pred, zero_division=0):.4f} "
          f"F1 {f1_score(y_test, pred, zero_division=0):.4f} | alerts {int(pred.sum())} TP {int((pred & y_test).sum())}/{int(y_test.sum())} "
          f"| test PR-AUC {average_precision_score(y_test, test_s):.4f}")
report(f"PCA ({best_var} var)", val_pca, test_pca)

   variance  components  val_pr_auc
0      0.50           8       0.623
1      0.60          10       0.624
2      0.70          12       0.626
3      0.80          14       0.651
4      0.90          16       0.652
5      0.95          18       0.666
PCA (0.95 var): P 0.9870 R 0.6609 F1 0.7917 | alerts 385 TP 380/575 | test PR-AUC 0.7003


In [152]:
   def pca_t2(p, Z): return np.sum(p.transform(Z) ** 2 / p.explained_variance_, axis=1)
   tr_spe, tr_t2 = pca_scores(pca, Xn), pca_t2(pca, Xn)
   comb = lambda Z: (robust(pca_scores(pca, Z), tr_spe) + robust(pca_t2(pca, Z), tr_t2)) / 2
   print("PCA recon only:", round(average_precision_score(y_test, test_pca), 4),
         "| recon + T2:", round(average_precision_score(y_test, comb(Xt)), 4))

PCA recon only: 0.7003 | recon + T2: 0.7003


In [153]:
import torch, torch.nn as nn, copy
from torch.utils.data import DataLoader, TensorDataset
from sklearn.cluster import Birch
from sklearn.metrics import pairwise_distances_argmin_min

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(20260930); np.random.seed(20260930)

class BAE(nn.Module):
    def __init__(self, d_in, hidden=32, latent=4):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, hidden), nn.ReLU(), nn.Linear(hidden, latent))
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.ReLU(), nn.Linear(hidden, d_in))
    def forward(self, x): return self.decoder(self.encoder(x))

Xn_t = torch.tensor(Xn, dtype=torch.float32, device=device)
Xv_norm_t = torch.tensor(Xv[~y_val], dtype=torch.float32, device=device)   # normal-only early stopping
bae = BAE(Xn.shape[1]).to(device)
opt = torch.optim.Adam(bae.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()
loader = DataLoader(TensorDataset(Xn_t), batch_size=256, shuffle=True)
best, best_state, wait = float("inf"), None, 0
for epoch in range(400):
    bae.train()
    for (b,) in loader:
        opt.zero_grad(); loss_fn(bae(b), b).backward(); opt.step()
    bae.eval()
    with torch.no_grad():
        v = loss_fn(bae(Xv_norm_t), Xv_norm_t).item()
    if v < best: best, best_state, wait = v, copy.deepcopy(bae.state_dict()), 0
    else:
        wait += 1
        if wait >= 15: break
bae.load_state_dict(best_state); bae.eval()
print(f"Stopped at epoch {epoch + 1}, best normal-validation loss {best:.4f}")

def bae_parts(Z):
    t = torch.tensor(Z, dtype=torch.float32, device=device)
    with torch.no_grad():
        lat, rec = bae.encoder(t), bae(t)
    return torch.mean((t - rec) ** 2, dim=1).cpu().numpy(), lat.cpu().numpy()

tr_rec, tr_lat = bae_parts(Xn); va_rec, va_lat = bae_parts(Xv); te_rec, te_lat = bae_parts(Xt)
lat_scaler = StandardScaler().fit(tr_lat)
tr_lat, va_lat, te_lat = (lat_scaler.transform(a) for a in (tr_lat, va_lat, te_lat))

def robust(v, ref):
    iqr = np.percentile(ref, 75) - np.percentile(ref, 25)
    if iqr <= 1e-12: iqr = np.std(ref) if np.std(ref) > 1e-12 else 1.0
    return (v - np.median(ref)) / iqr

def bae_scores(th):
    centers = Birch(threshold=th, n_clusters=None).fit(tr_lat).subcluster_centers_
    d = lambda z: pairwise_distances_argmin_min(z, centers)[1]
    tr_d = d(tr_lat)
    mk = lambda rec, lat: (robust(rec, tr_rec) + robust(d(lat), tr_d)) / 2
    return mk(va_rec, va_lat), mk(te_rec, te_lat), len(centers)

rows = []
for th in [0.5, 1.0, 1.5, 2.0, 3.0]:
    v, _, k = bae_scores(th)
    rows.append({"birch_threshold": th, "subclusters": k, "val_pr_auc": average_precision_score(y_val, v)})
bae_sweep = pd.DataFrame(rows); print(bae_sweep.round(3))
best_th = bae_sweep.loc[bae_sweep.val_pr_auc.idxmax(), "birch_threshold"]
val_bae, test_bae_scores, _ = bae_scores(best_th)
report("BAE", val_bae, test_bae_scores)

Stopped at epoch 395, best normal-validation loss 0.2831
   birch_threshold  subclusters  val_pr_auc
0              0.5          200       0.624
1              1.0           24       0.613
2              1.5            8       0.613
3              2.0            5       0.627
4              3.0            1       0.666
BAE: P 0.9294 R 0.6643 F1 0.7748 | alerts 411 TP 382/575 | test PR-AUC 0.6998


In [154]:
  print("Recon-only test PR-AUC:", round(average_precision_score(y_test, te_rec), 4))
  print("Combined test PR-AUC:  ", round(average_precision_score(y_test, test_bae_scores), 4))

Recon-only test PR-AUC: 0.6492
Combined test PR-AUC:   0.6998


In [155]:
rng_b = np.random.default_rng(0); diffs = []
for _ in range(1000):
    i = rng_b.integers(0, len(y_test), len(y_test))
    if y_test[i].sum() == 0: continue
    diffs.append(average_precision_score(y_test[i], test_bae_scores[i]) - average_precision_score(y_test[i], test_pca[i]))
print(np.percentile(diffs, [2.5, 50, 97.5]).round(3))

[-0.006 -0.001  0.005]


In [157]:
rule = (df["amount"] > df["sender_balance_before"]).to_numpy()[te]
print("Rule: P", round(precision_score(y_test, rule), 4), "R", round(recall_score(y_test, rule), 4), "F1", round(f1_score(y_test, rule), 4))

Rule: P 1.0 R 0.6122 F1 0.7594


In [156]:
from google.colab import drive
import os
import joblib

drive.mount('/content/drive')

drive_models_dir = "/content/drive/MyDrive/models"
os.makedirs(drive_models_dir, exist_ok=True)

torch.save(bae.state_dict(), f"{drive_models_dir}/bae.pth")
print(f"Autoencoder weights saved to: {drive_models_dir}/bae.pth")

safe = lambda x: x if x > 1e-12 else 1.0
final_centers = Birch(threshold=best_th, n_clusters=None).fit(tr_lat).subcluster_centers_
train_cluster_d = pairwise_distances_argmin_min(tr_lat, final_centers)[1]
val_s, _, _ = bae_scores(best_th)
p_, r_, t_ = precision_recall_curve(y_val, val_s)
f1_ = 2 * p_[:-1] * r_[:-1] / (p_[:-1] + r_[:-1] + 1e-12)

torch.save(bae.state_dict(), f"{drive_models_dir}/bae.pth")
bae_bundle = {
    "feature_scaler": scaler, "latent_scaler": lat_scaler, "birch_centers": final_centers,
    "threshold": float(t_[np.argmax(f1_)]), "feature_columns": list(keep),
    "stats": {
        "recon_median": float(np.median(tr_rec)),
        "recon_iqr": safe(float(np.percentile(tr_rec, 75) - np.percentile(tr_rec, 25))),
        "cluster_median": float(np.median(train_cluster_d)),
        "cluster_iqr": safe(float(np.percentile(train_cluster_d, 75) - np.percentile(train_cluster_d, 25))),
    },
}
joblib.dump(bae_bundle, f"{drive_models_dir}/bae_bundle.joblib")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Autoencoder weights saved to: /content/drive/MyDrive/models/bae.pth


['/content/drive/MyDrive/models/bae_bundle.joblib']

Bootstrap 95% interval for the PR-AUC difference (BAE minus PCA): −0.006 to 0.005 (includes zero, so the models are indistinguishable).

BIRCH ablation: reconstruction-only BAE PR-AUC 0.6492 vs 0.6998 for the combined score (+0.051). At the validation-selected threshold (3.0) BIRCH forms one subcluster, so the gain comes from latent-space distance to the normal centroid.